In [0]:
VECTOR_DB_PATH = "/Volumes/workspace/legal_data/vector_db-test/"

In [0]:
%pip install sentence-transformers chromadb

In [0]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

gold_df = spark.read.format("delta").load(GOLD_PATH)

gold_df = gold_df.select(
    "chunk_id",
    "chunk_text",
    "act_name",
    "section_number",
    "category",
    "file_name"
)

gold_df.display()

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [0]:
client = chromadb.Client(Settings(
    persist_directory=VECTOR_DB_PATH
))

collection = client.get_or_create_collection(
    name="legal_knowledge"
)

In [0]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)

In [0]:
batch_size = 100
rows = gold_df.collect()

for i in range(0, len(rows), batch_size):
    batch = rows[i:i+batch_size]

    texts = [str(r.chunk_text) for r in batch]
    ids = [str(r.chunk_id) for r in batch]

    embeddings = model.encode(texts).tolist()

    # ✅ sanitize metadata
    metadata = []
    for r in batch:
        metadata.append({
            "act_name": str(r.act_name) if r.act_name else "",
            "section": str(r.section_number) if r.section_number else "",
            "category": str(r.category) if r.category else "",
            "source": str(r.file_name) if r.file_name else ""
        })

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(f"Processed batch {i}")

In [0]:
query = "What is the penalty under Motor Vehicles Act for not wearing helmet under section 129?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=10
)

docs = results["documents"][0]

filtered_docs = [doc for doc in docs if "helmet" in doc.lower()]

print(filtered_docs[:3])